## Witaj w Drugim Labie - Tydzień 1, Dzień 3

Dziś popracujemy z mnóstwem modeli! To sposób, żeby oswoić się z API.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ważna kwestia - proszę przeczytać</h2>
            <span style="color:#ff7800;">Sposób, w jaki z Tobą współpracuję, może różnić się od innych kursów, które przerabiałeś. Wolę nie pisać kodu na Twoich oczach. Zamiast tego wykonuję Jupyter Laby, tak jak ten, i daję Ci intuicję tego, co się dzieje. Sugeruję, żebyś sam dokładnie to wykonał, <b>po</b> obejrzeniu wykładu. Dodawaj instrukcje print, żeby zrozumieć, co się dzieje, a potem wymyśl własne warianty. Zobacz Q37 w <a href="https://edwarddonner.com/avatar?q=37">FAQ</a>, jak skonfigurować osobny projekt na swoją pracę.<br/><br/>Jeśli masz czas, byłbym zachwycony, gdybyś zgłosił PR ze zmianami w folderze community_contributions - instrukcje w materiałach. Jeśli masz konto na Githubie, wykorzystaj je, żeby pokazać swoje warianty. To nie tylko cenna praktyka, ale też pokazuje Twoje umiejętności innym, w tym być może przyszłym klientom albo pracodawcom...<br/>A jeśli opublikujesz o tym post na LinkedIn i oznaczysz mnie, to włączę się, żeby wzmocnić Twoje osiągnięcie. Jeśli widzisz innych kursantów publikujących posty, im też daj wsparcie.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Zacznij od importów - poproś Cursor Agenta o wyjaśnienie każdego pakietu, którego nie znasz

import os  # dostęp do zmiennych środowiskowych (os.getenv)
import json  # parsowanie odpowiedzi JSON od sędziego
from dotenv import load_dotenv  # wczytanie kluczy API z pliku .env
from openai import OpenAI  # klient OpenAI - używany też jako uniwersalny klient dla endpointów kompatybilnych z OpenAI (DeepSeek, Gemini, Groq, Grok, OpenRouter, Ollama)
from anthropic import Anthropic  # natywny klient Anthropic (Messages API), inny kształt odpowiedzi niż przez interfejs OpenAI-compat
from IPython.display import Markdown, display  # renderowanie odpowiedzi modeli jako sformatowany Markdown w notatniku

In [ ]:
# Zawsze pamiętaj, żeby to zrobić!
load_dotenv(override=True)  # wczytuje .env i nadpisuje już ustawione zmienne środowiskowe

In [ ]:
# Wypisz prefiksy kluczy, żeby ułatwić ewentualny debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}") 
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:
# prompt proszący model o wygenerowanie trudnego pytania-testu dla innych LLM-ów
request = """
Wymyśl trudne, niejednoznaczne pytanie ze zwięzłą odpowiedzią,
które będę mógł zadać wielu LLM-om, żeby ocenić ich inteligencję.
Nie zagadka matematyczna, tylko pytanie skłaniające do myślenia, wymagające inteligentnego wglądu.
W treści pytania zaznacz, że odpowiedź musi być krótka.
"""
request += "Odpowiedz tylko pytaniem, bez wyjaśnienia."  # dokładamy do prompta instrukcję formatu odpowiedzi
messages = [{"role": "user", "content": request}]  # pakujemy prompt w listę messages (rola "user"), format wspólny dla OpenAI i Anthropic

In [ ]:
# podejrzyj zawartość messages przed wysłaniem do modelu
messages

In [ ]:
# Ta komórka MUSI się uruchomić - question zasila wszystkie kolejne komórki w notatniku,
# więc zamiast OpenAI (którego klucza celowo nie trzymam) używam tu Anthropic

claude = Anthropic()  # inicjalizacja natywnego klienta Anthropic (klucz brany automatycznie z ANTHROPIC_API_KEY)

response = claude.messages.create(model="claude-haiku-4-5", max_tokens=16000, messages=messages)  # max_tokens jest wymagany przez Anthropic API
question = next(block.text for block in response.content if block.type == "text")  # response.content to lista bloków (thinking/text/...), wyciągamy pierwszy tekstowy
display(Markdown(question))  # wyświetl wygenerowane pytanie jako sformatowany Markdown

## Wywoływanie LLM-ów od wielu dostawców

Zaraz wywołamy LLM-y od wielu różnych dostawców.
Wszyscy oni udostępniają endpointy API kompatybilne z OpenAI, co wyjaśniono w Guide 9 w folderze guides.
Możemy więc po prostu używać tych endpointów tak, jakbyśmy używali OpenAI.

Uwaga:

Użyję mnóstwa LLM-ów od różnych dostawców, ale Ty wcale nie musisz! To tylko po to, żeby pokazać ich możliwości.

In [ ]:
# Adresy URL kompatybilne z OpenAI

ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
GROK_BASE_URL = "https://api.x.ai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
# Klienci OpenAI z odpowiednim base_url i kluczem
# Jeśli Cię to zaskakuje, zobacz Guide 9 w folderze Guides!

anthropic = OpenAI(api_key=anthropic_api_key, base_url=ANTHROPIC_BASE_URL)  # UWAGA: to klient OpenAI-compat pod Anthropic, inny niż natywny `claude = Anthropic()` z komórki wyżej - celowo, bo to lab porównawczy wielu dostawców przez jeden wspólny interfejs
deepseek = OpenAI(api_key=deepseek_api_key, base_url=DEEPSEEK_BASE_URL)
gemini = OpenAI(api_key=google_api_key, base_url=GEMINI_BASE_URL)
groq = OpenAI(api_key=groq_api_key, base_url=GROQ_BASE_URL)
grok = OpenAI(api_key=grok_api_key, base_url=GROK_BASE_URL)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_BASE_URL)
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
competitors = []  # nazwy modeli, w kolejności zgłaszania odpowiedzi
answers = []  # odpowiedzi modeli, w tej samej kolejności co competitors
messages = [{"role": "user", "content": question}]  # to samo pytanie wysyłane do każdego konkurenta

In [ ]:
def record(model_name, answer):  # wspólna funkcja: zapisuje wynik konkurenta i od razu go wyświetla
    competitors.append(model_name)
    answers.append(answer)
    display(Markdown(answer))

In [ ]:
# API, które już znamy
# Reasoning effort może być none, low, medium, high albo xhigh
# Ten kod nie uruchomi się bez OPENAI_API_KEY w .env — celowo go nie trzymam, zostaje dla porównania z wywołaniami innych dostawców poniżej

model_name = "gpt-5.4-nano"  # najtańszy/najszybszy poziom GPT-5.4

response = openai.chat.completions.create(model=model_name, messages=messages, reasoning_effort="none")  # reasoning_effort steruje głębokością rozumowania modelu
answer = response.choices[0].message.content  # kształt odpowiedzi OpenAI: tekst w response.choices[0].message.content

record(model_name, answer)

In [ ]:
model_name = "claude-sonnet-4-6"  # tu Anthropic przez interfejs kompatybilny z OpenAI (klient `anthropic` z komórki wyżej), nie natywne Anthropic SDK

response = anthropic.chat.completions.create(model=model_name, messages=messages)  # ten sam kształt wywołania co dla OpenAI, dzięki kompatybilnemu endpointowi
answer = response.choices[0].message.content  # format odpowiedzi jak w OpenAI (response.choices[...]), nie response.content jak w natywnym SDK z komórki 7

record(model_name, answer)

In [ ]:
model_name = "gemini-3.1-flash-lite"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
model_name = "deepseek-v4-flash"

response = deepseek.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
model_name = "openai/gpt-oss-120b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

# tu bez record() - to samo robimy ręcznie, identyczny efekt
display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
model_name = "moonshotai/kimi-k2.6"

response = openrouter.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)


In [ ]:
# OpenRouter, ale tym razem surowe requests zamiast klienta OpenAI-compat —
# potrzebujemy pola reasoning_details, którego SDK OpenAI nie ma w typowanym modelu odpowiedzi

import requests
import time

model_name = "google/gemma-4-26b-a4b-it:free"

# Darmowe modele na OpenRouterze dzielą wspólną pulę requestów u dostawcy (tu: Google AI Studio) —
# HTTP 429 "temporarily rate-limited upstream" jest częste i zwykle mija po kilku sekundach
for attempt in range(3):  # do 3 prób - darmowa pula bywa chwilowo przeciążona (HTTP 429)
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {openrouter_api_key}",
            "Content-Type": "application/json",
        },
        json={
            "model": model_name,
            "messages": messages,
            "reasoning": {"enabled": True},
        },
    )
    data = response.json()  # błąd OpenRoutera przychodzi jako HTTP 200 z ciałem {"error": {...}}, nie jako wyjątek
    if "choices" in data:
        break  # sukces - wychodzimy z pętli retry
    if data.get("error", {}).get("code") == 429 and attempt < 2:
        time.sleep(5)  # chwila odczekania, zanim spróbujemy ponownie
        continue
    raise RuntimeError(f"OpenRouter zwrócił błąd (HTTP {response.status_code}): {data}")  # inny błąd niż 429 albo wyczerpane próby - przerywamy

message = data["choices"][0]["message"]  # struktura odpowiedzi identyczna jak w OpenAI Chat Completions
answer = message["content"]

record(model_name, answer)

## W kolejnej komórce użyjemy Ollamy

Ollama uruchamia lokalną usługę webową, która daje endpoint kompatybilny z OpenAI,  
i uruchamia modele lokalnie przy użyciu wydajnego kodu w C++.

Jeśli nie masz Ollamy, zainstaluj ją, wchodząc na https://ollama.com, klikając Download i postępując zgodnie z instrukcjami.

Po instalacji powinieneś móc wejść tutaj: http://localhost:11434 i zobaczyć komunikat "Ollama is running"

Może być konieczny restart Cursora (a może i reboot). Potem otwórz Terminal (control+\`) i uruchom `ollama serve`

Przydatne komendy Ollamy (uruchamiaj je w terminalu albo z wykrzyknikiem w tym notatniku):

`ollama pull <model_name>` pobiera model lokalnie  
`ollama ls` listuje wszystkie pobrane modele  
`ollama rm <model_name>` usuwa wskazany model z pobranych

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super ważne - zignoruj mnie na własne ryzyko!</h2>
            <span style="color:#ff7800;">Wiele modeli w Ollamie jest ZDECYDOWANIE za dużych dla Twojego domowego komputera. Koniecznie przeglądaj modele na stronie Ollamy. Staraj się używać modeli o rozmiarze 3GB lub mniej, chyba że wiesz lepiej; llama3.2 to świetny pierwszy wybór. Nie wybieraj modeli kończących się na :cloud; to coś innego (chmurowa usługa inferencji, jak Groq).
            </span>
        </td>
    </tr>
</table>

In [ ]:
!ollama pull llama3.2

In [ ]:
import requests
requests.get('http://localhost:11434').content  # sprawdzenie, czy lokalny serwer Ollamy odpowiada

In [ ]:
import requests
models = requests.get('http://localhost:11434/v1/models').json()  # lista modeli pobranych lokalnie w Ollamie (endpoint kompatybilny z OpenAI)
for model in models.get("data"):
    print(model.get("id"))  # wypisz same ID modeli

In [ ]:
model_name = "llama3.2:1b"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

record(model_name, answer)

In [ ]:
model_name = "gpt-oss:latest"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
model_name = "gemma4:latest"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# No to gdzie jesteśmy?

print(len(competitors))
print(competitors)
print(answers)


In [ ]:
# Warto umieć używać "zip"
for competitor, answer in zip(competitors, answers):  # zip paruje elementy o tym samym indeksie z obu list
    print(f"Competitor: {competitor}\n\n{answer}")


In [ ]:
# Połączmy to razem - zwróć uwagę na użycie "enumerate"

together = ""
for index, answer in enumerate(answers):  # enumerate daje indeks (do numeracji "Response from competitor N") razem z elementem
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [ ]:
# prompt dla LLM-sędziego: ocenia i rankinguje wszystkie odpowiedzi konkurentów
judge = f"""Oceniasz konkurs między {len(competitors)} konkurentami.
Każdy model otrzymał to pytanie:

{question}

Twoim zadaniem jest ocenić każdą odpowiedź pod względem jasności i siły argumentacji oraz uszeregować je od najlepszej do najgorszej.
Odpowiedz w formacie JSON, i tylko JSON, w następującym formacie:
{{"results": ["numer najlepszego konkurenta", "numer drugiego najlepszego konkurenta", "numer trzeciego najlepszego konkurenta", ...]}}

Oto odpowiedzi każdego konkurenta:

{together}

Teraz odpowiedz JSON-em z uszeregowaną kolejnością konkurentów, niczym więcej. Nie dodawaj formatowania markdown ani bloków kodu."""


In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]  # pakujemy prompt sędziego w messages, do wysłania do modelu-sędziego (Grok)

## A teraz czas na Groka!

Reklamowany jako "najbardziej prawdomówny duży model językowy na świecie".. więc użyjmy go jako naszego LLM-sędziego

In [ ]:
# Czas na osąd!
# Grok to "najbardziej prawdomówny duży model językowy na świecie."

model_name = "grok-4.3"

response = grok.chat.completions.create(model=model_name, messages=judge_messages)  # Grok pełni tu rolę sędziego oceniającego pozostałych konkurentów
results = response.choices[0].message.content  # oczekujemy czystego JSON-a (patrz instrukcja w prompcie judge)
print(results)  # podejrzyj surową odpowiedź przed parsowaniem


In [ ]:
# OK, zamieńmy to na wyniki!

results_dict = json.loads(results)  # parsujemy JSON zwrócony przez sędziego
ranks = results_dict["results"]  # lista numerów konkurentów w kolejności od najlepszego
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]  # numeracja sędziego jest 1-indeksowana, więc -1 do indeksu listy
    print(f"Rank {index+1}: {competitor}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ćwiczenie</h2>
            <span style="color:#ff7800;">Który wzorzec (albo wzorce) tutaj wykorzystano? Spróbuj zaktualizować to, żeby dodać kolejny wzorzec projektowy Agentic.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Implikacje komercyjne</h2>
            <span style="color:#00bfff;">Tego typu wzorce - wysłanie zadania do wielu modeli i ocena wyników -
            są powszechne tam, gdzie trzeba poprawić jakość odpowiedzi LLM. To podejście można powszechnie stosować
            w projektach biznesowych, gdzie kluczowa jest dokładność.
            </span>
        </td>
    </tr>
</table>

### Przykładowe rozwiązanie (niezależne od ćwiczenia powyżej)

Wzorce już obecne w tym notatniku: **Parallelization** (to samo pytanie trafia równolegle do wielu modeli od różnych dostawców) i **Evaluator** (LLM-sędzia, Grok, ocenia i rankinguje odpowiedzi). Dodaję kolejny wzorzec — **Evaluator-Optimizer**: biorę najsłabiej ocenioną odpowiedź, proszę sędziego o konkretną krytykę, a potem inny model poprawia odpowiedź na podstawie tej krytyki. Na końcu sędzia ocenia jeszcze raz, czy poprawka faktycznie pomogła.

In [ ]:
# Znajdź najsłabiej ocenionego konkurenta - ostatnia pozycja w rankingu sędziego

worst_number = ranks[-1]  # ostatnia pozycja rankingu sędziego = najsłabsza odpowiedź
worst_index = int(worst_number) - 1  # znów -1, bo numeracja sędziego jest 1-indeksowana
worst_competitor = competitors[worst_index]
worst_answer = answers[worst_index]

print(f"Najsłabiej oceniony: {worst_competitor}")

In [ ]:
# Poproś sędziego (Grok) o konkretną krytykę tej odpowiedzi

critique_prompt = f"""Oceniłeś już odpowiedzi na to pytanie:

{question}

Oto odpowiedź, którą uznałeś za najsłabszą:

{worst_answer}

Napisz zwięzłą, konkretną krytykę (2-3 zdania) - co dokładnie osłabia tę odpowiedź i co powinno się poprawić."""

critique_messages = [{"role": "user", "content": critique_prompt}]  # pakujemy prompt krytyki w messages

response = grok.chat.completions.create(model="grok-4.3", messages=critique_messages)  # ten sam sędzia (Grok), teraz proszony o uzasadnienie
critique = response.choices[0].message.content

display(Markdown(critique))  # pokaż krytykę w notatniku

In [ ]:
# Optimizer: model poprawia odpowiedź na podstawie krytyki sędziego
# Najtańszy model Anthropic, zgodnie z pułapem kosztowym tego repo

optimizer_prompt = f"""Oto pytanie:

{question}

Oto poprzednia odpowiedź:

{worst_answer}

Sędzia ocenił ją jako najsłabszą z tego powodu:

{critique}

Napisz poprawioną, silniejszą odpowiedź, która odpowiada na tę krytykę. Odpowiedz tylko poprawioną odpowiedzią, bez dodatkowych komentarzy."""

optimizer_messages = [{"role": "user", "content": optimizer_prompt}]  # pakujemy prompt poprawy w messages

response = anthropic.chat.completions.create(model="claude-haiku-4-5", messages=optimizer_messages)  # ten sam klient OpenAI-compat pod Anthropic co w komórce z konkurentami
improved_answer = response.choices[0].message.content

display(Markdown(improved_answer))  # pokaż poprawioną odpowiedź

In [ ]:
# Sędzia ocenia jeszcze raz: czy poprawiona odpowiedź faktycznie jest lepsza?

recheck_prompt = f"""Oceniałeś odpowiedzi na to pytanie:

{question}

Odpowiedź A (oryginalna, najsłabsza):
{worst_answer}

Odpowiedź B (poprawiona wersja po Twojej krytyce):
{improved_answer}

Która odpowiedź jest teraz lepsza, A czy B? Odpowiedz jednym słowem: "A" albo "B"."""

recheck_messages = [{"role": "user", "content": recheck_prompt}]  # sędzia porównuje oryginał (A) z poprawioną wersją (B)

response = grok.chat.completions.create(model="grok-4.3", messages=recheck_messages)
verdict = response.choices[0].message.content  # oczekujemy jednego słowa: "A" albo "B"

print(f"Sędzia po poprawce wybiera: {verdict.strip()}")  # .strip() na wypadek białych znaków wokół odpowiedzi